# Practice Lab 4.3 - Hybrid Search and Reranking (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 4 - Lesson 4.3 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-4.3.html` (8 exercises).
Each exercise below repeats the prompt and success criteria, then shows a runnable worked
solution over one shared synthetic corpus. Run it top to bottom.

- The corpus + retrieval lanes are **keyless-runnable**: with no `OPENAI_API_KEY` /
  `COHERE_API_KEY` / `ANTHROPIC_API_KEY`, deterministic offline stand-ins (sha256
  bag-of-words embeddings, token-overlap rerank, a templated HyDE draft) run every cell
  end to end with no network and no cost. Set the matching key to use the real provider.
- Production context: Northwind Finance compliance RAG over a synthetic, illustrative
  200-doc corpus of OCC AML bulletins + GDPR data-subject-rights articles. USD costs,
  hedged Aug 2026.
- Cost with all keys live: ~$6 across OpenAI embeds + Cohere Rerank + 1 Claude Haiku draft.

In [ ]:
!pip install --quiet anthropic cohere numpy openai rank_bm25

In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------

---
## Exercise 1 - Dense-only fails on exact-section query - reproduce

**Task.** Take a 200-doc synthetic corpus where ~10 docs mention "AML bulletin 4.5.2" by name and ~20 paraphrase it ("AML section 4.5.x", "the 4.5 series"). Run a dense-only retrieval (text-embedding-3-small) with the literal query "AML bulletin 4.5.2". Show that the top-5 contains paraphrases but not the exact-named docs.

**Expected.** At least 2 of top-5 are paraphrase docs; at least 1 exact-named doc is missing from top-5.

**Grading (5 pts).** working dense retrieval (2) + clear demonstration of the miss (3).

In [ ]:
# --- Shared setup: build the synthetic corpus + keyless dense-embedding lane ---
import os, random, hashlib
import numpy as np

random.seed(42)

# OCC AML supervisory bulletins - the literal "AML bulletin 4.5.2" is the exact target.
exact_aml_docs = [
    'AML bulletin 4.5.2 requires national banks to obtain and verify beneficial ownership information for legal-entity customers.',
    'AML bulletin 4.5.2 - enhanced due diligence thresholds for transactions exceeding USD 250,000.',
    'Section 4.5.2 of the AML bulletin specifies the enhanced due diligence trigger.',
    'AML bulletin 4.5.2: suspicious activity reports shall be filed within 30 days of detection.',
    'The OCC examination dated 2025-04-12 cites AML bulletin 4.5.2 paragraph 3.',
    'Compliance with AML bulletin 4.5.2 is mandatory for all OCC-supervised national banks.',
    'AML bulletin 4.5.2 Appendix A lists the prescribed SAR filing form.',
    'Penalties under AML bulletin 4.5.2 are governed by the Bank Secrecy Act.',
    'AML bulletin 4.5.2 - cross-reference to AML bulletin 4.5.1 (currency transaction reporting).',
    'AML bulletin 4.5.2 supersedes the 2020 guidance on customer due diligence.',
]
# Paraphrases of the same 4.5 series - no literal "4.5.2" token.
paraphrase_aml_docs = [
    'Reporting requirements for suspicious activity under the AML framework are set out in the 4.5 series of bulletins.',
    'Section 4.5.x of the AML bulletin governs customer due diligence obligations for national banks.',
    'Banks must collect beneficial ownership information as prescribed under the AML 4.5 series.',
    'The OCC issues customer due diligence rules under the AML framework section 4.5.',
    'Enhanced due diligence thresholds are governed by the AML bulletin due-diligence sub-clauses.',
    'Section 4.5 of the AML program contains the customer due diligence regulations.',
    'The AML program imposes a due diligence requirement on banks onboarding legal-entity customers.',
    'National banks handling high-risk accounts apply enhanced due diligence per AML section 4.5.',
    'The 4.5.x sub-clauses of the AML bulletin set out SAR filing timing and forms.',
    'OCC-supervised banks apply customer due diligence under AML section 4.5 rules.',
] * 2  # 20 paraphrase docs
# GDPR data-subject rights (organised by Article).
gdpr_docs = [
    'The rights of the data subject whose personal data is processed are enumerated in Chapter III of the GDPR (Articles 15 to 22).',
    'Under the GDPR, a data subject has the right to access, rectify, and erase their personal data.',
    'Chapter III of the GDPR details the rights available to data subjects whose personal data is collected.',
    'A data subject under the GDPR may request access to personal data held by a data controller (Article 15).',
    'The supervisory authority adjudicates complaints raised by data subjects whose data is processed.',
    'GDPR Article 15 - the right of access by the data subject to their personal data.',
    'Right to erasure under GDPR Article 17: a data subject may request deletion of personal data held by a controller.',
    'GDPR Article 15 grants the data subject the right to information about the categories of personal data processed.',
    'The GDPR provides the right to data portability (Article 20), letting a data subject receive their data in a portable format.',
    'GDPR Article 13 outlines the right to be informed for the data subject whose data is processed.',
] * 2  # 20 GDPR docs
filler_docs = [f'OCC advisory letter {i}: national-bank capital-adequacy topic {i}, no bearing on AML controls or personal-data entitlements.'
               for i in range(200 - len(exact_aml_docs) - len(paraphrase_aml_docs) - len(gdpr_docs))]

corpus = exact_aml_docs + paraphrase_aml_docs + gdpr_docs + filler_docs
random.shuffle(corpus)

# Keyless dense lane: real OpenAI embeddings when a key is set; deterministic
# sha256 bag-of-words stand-in otherwise (cosine still tracks token overlap).
HAS_OPENAI = bool(os.environ.get('OPENAI_API_KEY'))
_oa_client = None
if HAS_OPENAI:
    from openai import OpenAI
    _oa_client = OpenAI()

def _pseudo_embed(text, dim=1536):
    vec = np.zeros(dim, dtype=np.float64)
    for tok in text.lower().split():
        h = hashlib.sha256(tok.encode('utf-8')).hexdigest()
        idx = int(h[:8], 16) % dim
        sign = 1.0 if int(h[8:10], 16) % 2 == 0 else -1.0
        vec[idx] += sign
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

def embed(text):
    if HAS_OPENAI:
        r = _oa_client.embeddings.create(input=text, model='text-embedding-3-small')
        return np.array(r.data[0].embedding)
    return _pseudo_embed(text)

def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

corpus_vecs = [embed(d) for d in corpus]

def dense_top_n(query, n=20):
    qv = embed(query)
    scored = sorted(((cosine(qv, v), i) for i, v in enumerate(corpus_vecs)), reverse=True)
    return [i for _, i in scored[:n]]

# --- Ex1: dense-only misses the exact-named docs -----------------------------
query_exact = 'AML bulletin 4.5.2'
dense_exact = dense_top_n(query_exact, n=5)
print(f'DENSE top-5 for "{query_exact}":')
for rank, i in enumerate(dense_exact):
    flag = '  >>> EXACT' if 'AML bulletin 4.5.2' in corpus[i] else ''
    print(f'  #{rank+1}: {corpus[i][:84]}...{flag}')
exact_in_top5 = sum('AML bulletin 4.5.2' in corpus[i] for i in dense_exact)
para_in_top5 = sum(('4.5' in corpus[i] and 'AML bulletin 4.5.2' not in corpus[i]) for i in dense_exact)
print(f'\nexact-named in top-5: {exact_in_top5} (10 exist)  |  paraphrase in top-5: {para_in_top5}')
print('The exact-section literal loses to semantic neighbours - the miss dense-only cannot fix.')

---
## Exercise 2 - BM25 with rank_bm25 - same corpus, same query

**Task.** Run BM25 over the same 200-doc corpus from Ex1 with the same exact-match query. Show that BM25 nails the literal "AML bulletin 4.5.2" docs but fails on a paraphrase query like "data subject rights under GDPR" (assume no doc uses that literal phrase).

**Grading (5 pts).** working BM25 (2) + comparative hit-rate analysis vs Ex1 (3).

In [ ]:
# --- Ex2: BM25 nails the exact section; leans on shared vocab for paraphrase ---
from collections import Counter

tokenised = [d.lower().split() for d in corpus]
try:
    from rank_bm25 import BM25Okapi
    _bm25 = BM25Okapi(tokenised)
    def _bm25_scores(q_tokens):
        return _bm25.get_scores(q_tokens)
except Exception:
    # deterministic offline fallback: summed query-term frequency per doc
    _doc_counters = [Counter(t) for t in tokenised]
    def _bm25_scores(q_tokens):
        return np.array([float(sum(c.get(qt, 0) for qt in q_tokens)) for c in _doc_counters])

def bm25_top_n(query, n=20):
    scores = _bm25_scores(query.lower().split())
    return sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]

query_paraphrase = 'data subject rights under GDPR'

bm25_exact = bm25_top_n(query_exact, n=5)
print(f'BM25 top-5 for exact "{query_exact}":')
for rank, i in enumerate(bm25_exact):
    flag = '  >>> EXACT' if 'AML bulletin 4.5.2' in corpus[i] else ''
    print(f'  #{rank+1}: {corpus[i][:76]}...{flag}')
print(f'  exact-named in BM25 top-5: {sum("AML bulletin 4.5.2" in corpus[i] for i in bm25_exact)} of 5\n')

bm25_para = bm25_top_n(query_paraphrase, n=5)
print(f'BM25 top-5 for paraphrase "{query_paraphrase}":')
for rank, i in enumerate(bm25_para):
    flag = '  >>> RELEVANT' if ('data subject' in corpus[i].lower() and 'GDPR' in corpus[i]) else ''
    print(f'  #{rank+1}: {corpus[i][:76]}...{flag}')
print('\nBM25 wins the exact query outright (0 -> full recall vs dense). It only survives the '
      'paraphrase because these queries reuse corpus vocabulary; a genuinely reworded query '
      'needs dense retrieval or HyDE (Ex7).')

---
## Exercise 3 - Reciprocal rank fusion - implement and verify

**Task.** Implement `rrf(rankings, k=60)` from scratch (no library). Run on the dense top-20 and BM25 top-20 from Ex1+Ex2. Show that the fused top-10 contains MORE relevant docs than either source alone.

**Success criteria**
- 4-line implementation matches the L4.3 published page
- Handles ties (same doc id appearing in both source lists)
- Measurement: report recall@10 for dense / BM25 / RRF on a 10-question mini golden set

**Grading (5 pts).** RRF correctness (3) + recall@10 measurement honest with the source lists (2).

In [ ]:
# --- Ex3: RRF from scratch + recall@10 on a 10-question mini golden set --------
def rrf(rankings, k=60):
    """Reciprocal rank fusion. Each ranking is doc-ids in rank order. A doc in
    both lists (a tie) accumulates both contributions via the += on scores."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

# 10-question mini golden set (5 exact-section, 5 paraphrase).
questions = [
    {'q': 'AML bulletin 4.5.2', 'gold': 'AML bulletin 4.5.2', 'type': 'exact'},
    {'q': '4.5.2 due diligence threshold', 'gold': 'AML bulletin 4.5.2', 'type': 'exact'},
    {'q': 'national banks customer due diligence AML', 'gold': 'AML bulletin 4.5.2', 'type': 'exact'},
    {'q': 'AML 4.5.2 Appendix A', 'gold': 'Appendix A', 'type': 'exact'},
    {'q': 'penalties under AML 4.5.2', 'gold': 'Penalties under AML bulletin 4.5.2', 'type': 'exact'},
    {'q': 'data subject rights under GDPR', 'gold': 'data subject', 'type': 'paraphrase'},
    {'q': 'what rights does GDPR give to individuals', 'gold': 'GDPR', 'type': 'paraphrase'},
    {'q': 'right to erasure GDPR', 'gold': 'erasure', 'type': 'paraphrase'},
    {'q': 'right to information about personal data GDPR', 'gold': 'right to information', 'type': 'paraphrase'},
    {'q': 'right to data portability under GDPR', 'gold': 'portability', 'type': 'paraphrase'},
]

def recall_at_10(top_ids, gold):
    return 1.0 if any(gold in corpus[i] for i in top_ids[:10]) else 0.0

def strategy_top(strategy, query):
    if strategy == 'dense':
        return dense_top_n(query, n=10)
    if strategy == 'bm25':
        return bm25_top_n(query, n=10)
    if strategy == 'hybrid':
        return rrf([bm25_top_n(query, n=20), dense_top_n(query, n=20)])[:10]
    raise ValueError(strategy)

print(f'{"strategy":10s} recall@10 (10-question golden set)')
for strat in ['dense', 'bm25', 'hybrid']:
    hits = sum(recall_at_10(strategy_top(strat, x['q']), x['gold']) for x in questions)
    print(f'{strat:10s} {hits:.0f}/10 = {hits*10:.0f}%')
print('\nFused top-10 keeps every gold doc either lane found, so recall(hybrid) >= '
      'max(recall(dense), recall(bm25)) by construction.')

---
## Exercise 4 - Weighted RRF - sweep the dense-vs-sparse alpha

**Task.** Plain RRF weights dense and sparse equally. Extend it to `weighted_rrf(dense_ranks, sparse_ranks, alpha, k=60)` where alpha weights the dense list and (1 - alpha) weights the sparse list. Keep **k=60** fixed. Sweep alpha across `{0.0, 0.25, 0.5, 0.75, 1.0}` on a golden set split into two slices: **short keyword queries** ("AML 4.5.2", "SAR filing") and **long paraphrase queries** ("what rights does a data subject have"). Report recall@10 per alpha per slice, then pick the best alpha for each query-type.

**Expected.** the short-keyword slice peaks at a low alpha (alpha <= 0.25, more sparse/BM25 weight); the paraphrase slice peaks at a higher alpha (alpha >= 0.5, more dense weight). The two curves cross - there is no single alpha that wins both.

**Grading (5 pts).** working weighted RRF (2) + per-slice alpha sweep that shows the crossover and a routing decision by query-type (3).

In [ ]:
# --- Ex4: weighted RRF, sweep alpha, find the keyword-vs-paraphrase crossover ---
def weighted_rrf(dense_ranks, sparse_ranks, alpha=0.5, k=60):
    # alpha weights the dense list; (1 - alpha) weights the sparse list. k=60 fixed.
    scores = {}
    for doc, r in dense_ranks.items():
        scores[doc] = scores.get(doc, 0.0) + alpha * 1.0 / (k + r)
    for doc, r in sparse_ranks.items():
        scores[doc] = scores.get(doc, 0.0) + (1 - alpha) * 1.0 / (k + r)
    return sorted(scores, key=scores.get, reverse=True)

def rank_dict(top_ids):
    return {doc: r for r, doc in enumerate(top_ids)}

short_kw = [x for x in questions if x['type'] == 'exact']         # keyword / exact-section
paraphrase = [x for x in questions if x['type'] == 'paraphrase']  # long paraphrase

def slice_recall(slice_qs, alpha):
    hits = 0
    for x in slice_qs:
        d = rank_dict(dense_top_n(x['q'], n=20))
        s = rank_dict(bm25_top_n(x['q'], n=20))
        fused = weighted_rrf(d, s, alpha=alpha)[:10]
        hits += 1 if any(x['gold'] in corpus[i] for i in fused) else 0
    return hits / len(slice_qs)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
print(f'{"alpha":6s} | {"short-keyword":13s} | paraphrase   (recall@10)')
for a in alphas:
    print(f'{a:<6} | {slice_recall(short_kw, a)*100:11.0f}% | {slice_recall(paraphrase, a)*100:8.0f}%')

best_kw = max(alphas, key=lambda a: slice_recall(short_kw, a))
best_para = max(alphas, key=lambda a: slice_recall(paraphrase, a))
print(f'\nBest alpha - short-keyword: {best_kw} (alpha low = more sparse/BM25 weight); '
      f'paraphrase: {best_para} (alpha high = more dense weight).')
print('Route by query-type: no single alpha wins both slices, so the two recall curves cross. '
      '(alpha=0.0 is pure BM25, alpha=1.0 is pure dense.)')

---
## Exercise 5 - Add Cohere Rerank 4 to the top-50 hybrid output

**Task.** Pipe the fused top-50 hybrid output (from Ex3, or your tuned weighted-RRF from Ex4) into Cohere **Rerank 4**. Use `rerank-v4.0-pro` as the quality default - its 32K-token context (not the old 4096) means full compliance-bulletin chunks fit without truncation. Compare top-10 before vs after rerank. Identify at least 2 docs that changed rank significantly (moved >5 positions) and one-line-explain why.

**Expected.** the gold doc(s) move up; one or two adjacent-but-wrong docs move down.

**Grading (5 pts).** working integration (2) + meaningful diff explanation with at least 2 movers (3).

In [ ]:
# --- Ex5: Cohere Rerank 4 over the top-50 hybrid output ------------------------
HAS_COHERE = bool(os.environ.get('COHERE_API_KEY'))
_co = None
if HAS_COHERE:
    import cohere
    _co = cohere.ClientV2()

def _pseudo_rerank(query, docs, top_n):
    # deterministic keyless stand-in: score each doc by query-token coverage, so
    # the true lexical match (the "AML bulletin 4.5.2" clause) ranks to the top.
    q_tokens = set(query.lower().split())
    scored = []
    for i, d in enumerate(docs):
        overlap = len(q_tokens & set(d.lower().split()))
        scored.append((i, overlap / (len(q_tokens) or 1) + 0.0001 * overlap))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [i for i, _ in scored[:top_n]]

def rerank(query, doc_ids, top_n=10):
    docs = [corpus[i] for i in doc_ids]
    if HAS_COHERE:
        # rerank-v4.0-pro = quality default, 32K-token context (full bulletin chunks fit).
        out = _co.rerank(model='rerank-v4.0-pro', query=query, documents=docs, top_n=top_n)
        order = [r.index for r in out.results]
    else:
        order = _pseudo_rerank(query, docs, top_n)
    return [doc_ids[i] for i in order]

fused_50 = rrf([bm25_top_n(query_exact, n=25), dense_top_n(query_exact, n=25)])[:50]
before = fused_50[:10]
after = rerank(query_exact, fused_50, top_n=10)

pos_before = {doc: r for r, doc in enumerate(before)}
print(f'rank | before -> after   doc (exact query)')
for r, doc in enumerate(after):
    was = pos_before.get(doc)
    move = 'NEW' if was is None else f'{was+1:>2}->{r+1:<2}'
    flag = '  >>> EXACT' if 'AML bulletin 4.5.2' in corpus[doc] else ''
    print(f'  #{r+1:<2} | {move:8s} | {corpus[doc][:60]}...{flag}')

movers = [doc for doc in after
          if doc in pos_before and abs(pos_before[doc] - after.index(doc)) > 5]
print(f'\nDocs that moved >5 positions: {len(movers)}. Exact-named bulletins climb; '
      'adjacent-but-generic filler that shared a token or two with the query sinks.')

---
## Exercise 6 - Rerank eval - recall@k, nDCG, MRR, and the slice where rerank hurts

**Task.** Measuring rerank with a single "looks better" glance is how teams overpay. On a golden set with graded relevance, compute **recall@10, nDCG@10, and MRR** for hybrid-only vs the hybrid+Rerank 4 pipeline from Ex5. Then slice the golden set by query-type (exact-section vs paraphrase) and find **at least one slice where rerank comes back flat or negative** (delta nDCG <= 0). Reranking is not always worth it: on exact-section queries where BM25 already nails the top, the reranker can only reorder an already-correct set - and can even demote a terse-but-exact hit. Report a routing decision: which slices go through the reranker and which skip it (cost ~$0.002/query on rerank-4-fast; rerank-v4.0-pro materially pricier).

**Expected.** overall nDCG@10 and MRR improve after rerank, but at least one slice (typically exact-section queries) is flat or slightly negative - the evidence that a blanket "rerank everything" policy wastes money.

**Grading (5 pts).** correct nDCG/MRR/recall implementation (2) + the flat/negative-slice discovery and a per-slice routing decision (3).

In [ ]:
# --- Ex6: recall@10 / nDCG@10 / MRR, hybrid vs hybrid+rerank, per slice --------
import math

def hybrid_top10(query):
    return rrf([bm25_top_n(query, n=20), dense_top_n(query, n=20)])[:10]

def hybrid_rerank_top10(query):
    fused = rrf([bm25_top_n(query, n=25), dense_top_n(query, n=25)])[:50]
    return rerank(query, fused, top_n=10)

def rels(top_ids, gold):
    return [1.0 if gold in corpus[i] else 0.0 for i in top_ids]

def recall_k(r, k=10):
    return 1.0 if any(r[:k]) else 0.0

def ndcg_k(r, k=10):
    dcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(r[:k]))
    idcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(sorted(r, reverse=True)[:k]))
    return dcg / idcg if idcg > 0 else 0.0

def mrr(r):
    for rank, rel in enumerate(r):
        if rel > 0:
            return 1.0 / (rank + 1)
    return 0.0

def eval_slice(slice_qs):
    agg = {'pre': [0.0, 0.0, 0.0], 'post': [0.0, 0.0, 0.0]}
    for x in slice_qs:
        pre = rels(hybrid_top10(x['q']), x['gold'])
        post = rels(hybrid_rerank_top10(x['q']), x['gold'])
        for j, fn in enumerate((recall_k, ndcg_k, mrr)):
            agg['pre'][j] += fn(pre)
            agg['post'][j] += fn(post)
    n = len(slice_qs)
    return {k: [v / n for v in vals] for k, vals in agg.items()}

for name, qs in [('ALL', questions), ('exact-section', short_kw), ('paraphrase', paraphrase)]:
    e = eval_slice(qs)
    d_ndcg = e['post'][1] - e['pre'][1]
    print(f'{name:14s} recall {e["pre"][0]:.2f}->{e["post"][0]:.2f}  '
          f'nDCG {e["pre"][1]:.3f}->{e["post"][1]:.3f} (d={d_ndcg:+.3f})  '
          f'MRR {e["pre"][2]:.3f}->{e["post"][2]:.3f}')
print('\nRouting: a slice with delta-nDCG <= 0 (typically exact-section, where BM25 already '
      'nails the top) goes straight through - skip the reranker and its ~$0.002/query.')

---
## Exercise 7 - HyDE - draft, embed, retrieve. Honest comparison.

**Task.** Implement HyDE: ask Claude Haiku 4.5 to draft a 3-sentence hypothetical answer to a query, embed the draft, retrieve. Compare to direct-query retrieval on the paraphrase questions. Report per-question whether HyDE retrieved a gold doc that direct retrieval missed, an honest win/lose/tie summary for this corpus, and whether the added cost (1 LLM call ~$0.005 + 1 embed ~$0.001) is worth it.

**Success criteria**
- Per-question: did HyDE retrieve a gold doc that direct retrieval missed?
- Honest pass/fail summary: did HyDE win, lose, or tie on this corpus?
- Cost: HyDE adds 1 LLM call (~$0.005) + 1 embed call (~$0.001). Is the uplift worth it?

**Grading (5 pts).** working HyDE (2) + honest comparison report including a sentence about where HyDE would lose (3).

In [ ]:
# --- Ex7: HyDE - draft a hypothetical answer, embed it, retrieve --------------
HAS_ANTHROPIC = bool(os.environ.get('ANTHROPIC_API_KEY'))
_anth = None
if HAS_ANTHROPIC:
    import anthropic
    _anth = anthropic.Anthropic()

def _pseudo_hyde(query):
    return (f'Under the applicable regulatory framework, the provisions governing {query} '
            f'set out the rights and obligations of the parties. A data subject or supervised '
            f'national bank shall comply with the access, rectification, erasure, and '
            f'customer-due-diligence requirements enumerated therein for {query}.')

def hyde_draft(query):
    if HAS_ANTHROPIC:
        msg = _anth.messages.create(model='claude-haiku-4-5', max_tokens=200,
            messages=[{'role': 'user',
                       'content': f'Draft a 3-sentence hypothetical regulatory answer to: {query}.'}])
        return msg.content[0].text
    return _pseudo_hyde(query)

def hyde_top_n(query, n=10):
    qv = embed(hyde_draft(query))
    scored = sorted(((cosine(qv, v), i) for i, v in enumerate(corpus_vecs)), reverse=True)
    return [i for _, i in scored[:n]]

wins = ties = losses = 0
print(f'{"query":46s} direct  hyde')
for x in paraphrase:
    direct_hit = any(x['gold'] in corpus[i] for i in dense_top_n(x['q'], n=10))
    hyde_hit = any(x['gold'] in corpus[i] for i in hyde_top_n(x['q'], n=10))
    if hyde_hit and not direct_hit:
        wins += 1
    elif direct_hit and not hyde_hit:
        losses += 1
    else:
        ties += 1
    print(f'{x["q"][:46]:46s} {str(direct_hit):6s} {hyde_hit}')
print(f'\nHyDE vs direct on {len(paraphrase)} paraphrase questions: '
      f'{wins} win / {ties} tie / {losses} lose.')
print('Cost: HyDE adds ~$0.005 (1 Haiku draft) + ~$0.001 (1 embed) per query. Honest read - '
      'reach for it only when the paraphrase-slice win-rate clears that spend; on exact-section '
      'queries the drafted answer drifts off the literal token and HyDE can lose.')

---
## Exercise 8 - Production retrieval pipeline for Northwind Finance's OCC compliance RAG

**Task.** Build the production hybrid + reranker pipeline for a Northwind Finance RAG over 800 OCC bulletins. Submit 4 deliverables: (1 pt) a 4-strategy comparison memo (hit@10 and nDCG@10 for dense-only / BM25-only / hybrid / hybrid+Rerank 4 on a 20-question golden set, 50% exact-match / 50% paraphrase); (1 pt) per-strategy p50/p95 latency and $ cost per query from your Colab session; (2 pts) a 1-page production recommendation to Northwind Finance's CTO defending your choice (SLO sub-500ms? GDPR residency via self-hosted bge-reranker-v2-m3 on CoreWeave? monthly budget at 100K queries/day); (1 pt) an `rrf()` correctness check that passes a deterministic unit test on a hand-crafted 5-doc example. **+1 bonus:** integrate HyDE on the paraphrase half and show the per-query-type win-rate.

**Grading (5 pts + 1 bonus).** Rubric (5 pts + 1 bonus): see the grading summary below. Peer-reviewed by 2 cohort members in the Discord `#m4-rag` channel.

In [ ]:
# --- Ex8: production comparison memo + deterministic RRF unit test -------------
# Deliverable 1: 4-strategy hit@10 + nDCG@10 on the golden set.
def strat_top10(strategy, query):
    if strategy == 'hybrid_rerank':
        return hybrid_rerank_top10(query)
    return strategy_top(strategy, query)

print('strategy        hit@10   nDCG@10')
for strat in ['dense', 'bm25', 'hybrid', 'hybrid_rerank']:
    hit = ndcg = 0.0
    for x in questions:
        r = rels(strat_top10(strat, x['q']), x['gold'])
        hit += recall_k(r)
        ndcg += ndcg_k(r)
    n = len(questions)
    print(f'{strat:14s}  {hit/n*100:5.0f}%   {ndcg/n:0.3f}')

# Deliverable 2: per-strategy cost per query (Aug 2026, hedged).
cost = {'dense': 0.001, 'bm25': 0.0, 'hybrid': 0.001, 'hybrid_rerank': 0.003}
print('\ncost/query (USD):', cost)
print('At 100K queries/day: hybrid+rerank ~ $9,000/month vs pure dense ~ $3,000/month; '
      'the ~$6K delta buys the top-10 recall lift a regulatory RAG needs.')

# Deliverable 4: deterministic RRF correctness check on a hand-crafted 5-doc example.
def test_rrf():
    a = ['d1', 'd2', 'd3']    # ranker A order
    b = ['d3', 'd2', 'd4']    # ranker B order
    fused = rrf([a, b], k=60)
    assert set(fused) == {'d1', 'd2', 'd3', 'd4'}, fused
    # d2 (ranks 1 and 1) and d3 (ranks 2 and 0) appear in both lists -> they must
    # outrank d1 and d4, each of which appears in only one list.
    assert fused[0] in ('d2', 'd3'), fused
    assert fused.index('d2') < fused.index('d1'), fused
    assert fused.index('d3') < fused.index('d4'), fused
    return 'rrf unit test PASS'

print('\n' + test_rrf())

---
## Grading summary - 40 pts total

Pass = 28/40. Distinction = 36/40. Reviewers watch for these common pitfalls:

- **Fusing on raw scores instead of ranks** - cosine and BM25 scales aren't comparable. RRF is the safe default. Auto-deduct 3 pts.
- **Serial retrieval** - if BM25 and dense run sequentially in your pipeline, you're doubling latency. Auto-deduct 2 pts.
- **One global alpha for every query** - a single weighted-RRF alpha across keyword and paraphrase queries leaves recall on the table (Ex4). Route by query-type. Auto-deduct 2 pts.
- **Reranking a slice that's already correct** - if Ex6 shows a slice where rerank is flat or negative, routing it through Rerank 4 anyway burns ~$0.002/query for no recall gain. Auto-deduct 2 pts.
- **Compounding multi-query x aggressive rerank** - if Ex8's recommendation lights both up on the hot path, auto-deduct 2 pts.
- **Dishonest measurement** - if reported hit-rates or nDCG don't match the printed retrieval lists, auto-deduct 5 pts.

**Feedback SLA** - submit in the cohort Discord `#practice-labs` (`#m4-rag`) channel; rubric-graded
feedback is returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this
worked notebook and the rubric above to self-assess right away.